BINANCE

In [ ]:
pip install binance

     -------------------------------------- 851.8/851.8 kB 7.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


FOR HOURLY OHLCV

In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time 
import os 

def fetch_binance_klines(symbol, interval, start_time, end_time, limit=1000):
    url = 'https://api.binance.com/api/v3/klines'
    data = []
    while start_time < end_time:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': int(start_time.timestamp() * 1000),
            'endTime': int(end_time.timestamp() * 1000),
            'limit': limit
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"Error fetching data: {response.status_code}")
            break
        klines = response.json()
        if not klines:
            break
        data.extend(klines)
        last_open_time = klines[-1][0] / 1000
        start_time = datetime.fromtimestamp(last_open_time + 3600)  # Increment by 1 hour
        time.sleep(0.5)  # Sleep to respect API rate limits
    return data

# Define parameters
symbol = 'BTCUSDT'
interval = '1h'
end_time = datetime.utcnow()
start_time = end_time - timedelta(days=365)

# Fetch data
klines = fetch_binance_klines(symbol, interval, start_time, end_time)

# Convert to DataFrame
columns = ['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume',
           'Close Time', 'Quote Asset Volume', 'Number of Trades',
           'Taker Buy Base Asset Volume', 'Taker Buy Quote Asset Volume', 'Ignore']
df = pd.DataFrame(klines, columns=columns)

# Convert timestamps to datetime
df['Open Time'] = pd.to_datetime(df['Open Time'], unit='ms')
df['Close Time'] = pd.to_datetime(df['Close Time'], unit='ms')

# Save to CSV 
df.drop(columns=['Close Time', 'Taker Buy Quote Asset Volume', 'Number of Trades','Ignore'],inplace=True) 


In [3]:
df.dtypes

Open Time                      datetime64[ns]
Open                                   object
High                                   object
Low                                    object
Close                                  object
Volume                                 object
Quote Asset Volume                     object
Taker Buy Base Asset Volume            object
dtype: object

In [4]:
numeric_columns = [
    'Open', 'High', 'Low', 'Close',
    'Volume', 'Quote Asset Volume', 'Taker Buy Base Asset Volume'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
df.dtypes

Open Time                      datetime64[ns]
Open                                  float64
High                                  float64
Low                                   float64
Close                                 float64
Volume                                float64
Quote Asset Volume                    float64
Taker Buy Base Asset Volume           float64
dtype: object

In [6]:
start_date = datetime(2024, 6, 1)
end_date = datetime(2025, 4, 11)

df = df[(df['Open Time'] >= start_date) & (df['Open Time'] <= end_date)]
df.reset_index(drop=True, inplace=True)

In [7]:
df.rename(columns={
    'Open Time': 'datetime',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Quote Asset Volume': 'quote_volume',
    'Taker Buy Base Asset Volume': 'taker_buy_volume'
}, inplace=True)


In [8]:
output_dir = os.path.join("..","final_datasets")
output_file = os.path.join(output_dir, "BTCUSDT_hourly_data_Binance.csv")

df.to_csv(output_file, index=False)
print("Data saved to BTCUSDT_hourly_data_Binance.csv")

Data saved to BTCUSDT_hourly_data_Binance.csv


FOR DAILY OHLCV 

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import time 
import os 

def fetch_binance_klines(symbol, interval, start_time, end_time, limit=1000):
    url = 'https://api.binance.com/api/v3/klines'
    data = []

    while start_time < end_time:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': int(start_time.timestamp() * 1000),
            'endTime': int(end_time.timestamp() * 1000),
            'limit': limit
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"Error fetching data: {response.status_code}")
            break
        klines = response.json()
        if not klines:
            break
        data.extend(klines)
        
        # Advance by the correct amount (1d interval = 86400 sec)
        last_open_time = klines[-1][0] / 1000
        start_time = datetime.fromtimestamp(last_open_time + 86400, tz=timezone.utc)
        time.sleep(0.5)  # Respect rate limit

    return data

# Define parameters
symbol = 'BTCUSDT'
interval = '1d'
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=365)

# Fetch data
klines = fetch_binance_klines(symbol, interval, start_time, end_time)

# Convert to DataFrame
columns = ['Open Time', 'Open', 'High', 'Low', 'Close', 'Volume',
           'Close Time', 'Quote Asset Volume', 'Number of Trades',
           'Taker Buy Base Asset Volume', 'Taker Buy Quote Asset Volume', 'Ignore']
df_1d = pd.DataFrame(klines, columns=columns)

# Convert timestamps to datetime
df_1d['Open Time'] = pd.to_datetime(df_1d['Open Time'], unit='ms')
df_1d['Close Time'] = pd.to_datetime(df_1d['Close Time'], unit='ms')

# Save to CSV 
df_1d.drop(columns=['Close Time', 'Taker Buy Quote Asset Volume','Ignore'],inplace=True) 


In [15]:
df_1d.dtypes

Open Time                      datetime64[ns]
Open                                   object
High                                   object
Low                                    object
Close                                  object
Volume                                 object
Quote Asset Volume                     object
Number of Trades                        int64
Taker Buy Base Asset Volume            object
dtype: object

In [16]:
start_date = datetime(2024, 6, 1)
end_date = datetime(2025, 4, 11)

df_1d = df_1d[(df_1d['Open Time'] >= start_date) & (df_1d['Open Time'] <= end_date)]
df_1d.reset_index(drop=True, inplace=True)

In [17]:
numeric_columns = [
    'Open', 'High', 'Low', 'Close', 'Number of Trades',
    'Volume', 'Quote Asset Volume', 'Taker Buy Base Asset Volume'
]

for col in numeric_columns:
    df_1d[col] = pd.to_numeric(df_1d[col], errors='coerce')

df_1d.rename(columns={
    'Open Time': 'date',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Quote Asset Volume': 'quote_volume',
    'Taker Buy Base Asset Volume': 'taker_buy_volume',
    'Number of Trades': 'number_of_trades'
}, inplace=True)

In [18]:
df_1d.dtypes

date                datetime64[ns]
open                       float64
high                       float64
low                        float64
close                      float64
volume                     float64
quote_volume               float64
number_of_trades             int64
taker_buy_volume           float64
dtype: object

In [19]:
output_dir = os.path.join("..","final_datasets")
output_file = os.path.join(output_dir, "BTCUSDT_daily_data_Binance.csv")

df_1d.to_csv(output_file, index=False)
print("Data saved to BTCUSDT_daily_data_Binance.csv")

Data saved to BTCUSDT_daily_data_Binance.csv
